# CekDulu: fine-tune IndoBERT di Google Colab

Runtime → Change runtime type → **T4 GPU**. Notebook ini memakai split data yang sama dengan baseline sehingga hasilnya dapat dibandingkan secara adil.

1. Unggah `cekdulu.zip` (repo ini) ke Colab **atau** clone dari GitHub Anda.
2. Jalankan sel berurutan.
3. Unduh `models/indobert` dan pakai dengan `CEKDULU_CLASSIFIER_BACKEND=transformers`.

In [ ]:
# Opsi 1: unggah zip repo
from google.colab import files
up = files.upload()  # pilih cekdulu.zip
!unzip -q -o cekdulu.zip -d /content && ls /content/cekdulu
# Opsi 2: !git clone https://github.com/<username>/cekdulu.git /content/cekdulu

In [ ]:
%cd /content/cekdulu
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -r requirements-train-indobert.txt && pip -q install -e .

In [ ]:
# Reproduksi split (memverifikasi SHA-256 dataset) lalu baseline sebagai pembanding
!python scripts/prepare_data.py
!python scripts/train_baseline.py

In [ ]:
# Fine-tune IndoBERT (≈ beberapa menit di T4)
!python scripts/train_indobert.py --config configs/indobert.yaml

In [ ]:
# Bandingkan pada test set yang sama dan challenge set OOD
import json
b = json.load(open('reports/baseline/metrics.json'))
i = json.load(open('reports/indobert/metrics.json'))
bk = [k for k in b['test'] if k.startswith('word_char')][0]
for name, t in [('baseline', b['test'][bk]['tuned_threshold']), ('indobert', i['test']['tuned_threshold'])]:
    f = t['per_class']['penipuan']
    print(f"{name:9s} macroF1={t['macro_f1']:.3f} fraud P={f['precision']:.3f} R={f['recall']:.3f}")
print('CI indobert:', i['test']['bootstrap_95ci_tuned'])
!python scripts/evaluate_ood.py --backend transformers --model-dir models/indobert --out reports/indobert/ood_metrics.json

**Interpretasi:** dengan n=171 di test set, selisih macro-F1 di bawah ±0,03 umumnya berada di dalam interval kepercayaan. Klaim "IndoBERT lebih baik" hanya jika CI dan hasil OOD sama-sama mendukung. Pertimbangkan juga biaya: IndoBERT-base ±440 MB dan latensi CPU jauh di atas TF-IDF+LR (±2 ms).

In [ ]:
!zip -qr indobert_model.zip models/indobert -x 'models/indobert/checkpoints/*' reports/indobert
files.download('indobert_model.zip')